In [4]:
# !uv add scikit-learn

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [6]:
iris = load_iris()

In [11]:
X, y = iris.data, iris.target

In [14]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, test_size=0.2)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

In [18]:
class IrisNet(nn.Module):


    def __init__(self):

        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )


    def forward(self, X):

        return self.model(X)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── 1. Load & prep data ──────────────────────────────────────────
iris = load_iris()
X, y = iris.data, iris.target          # X: (150, 4)  y: 0/1/2

scaler = StandardScaler()
X = scaler.fit_transform(X)            # zero mean, unit variance

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# numpy → torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)    # long for CrossEntropyLoss
y_test  = torch.tensor(y_test,  dtype=torch.long)

# ── 2. Model ─────────────────────────────────────────────────────
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16),     # 4 features → 16 hidden
            nn.ReLU(),
            nn.Linear(16, 16),    # 16 → 16
            nn.ReLU(),
            nn.Linear(16, 3)      # 16 → 3 classes (raw logits)
        )

    def forward(self, x):
        return self.net(x)        # no softmax here — CrossEntropyLoss handles it

model = IrisNet()

# ── 3. Loss & optimizer ──────────────────────────────────────────
criterion = nn.CrossEntropyLoss()          # expects raw logits + class indices
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ── 4. Training loop ─────────────────────────────────────────────
for epoch in range(200):
    model.train()

    logits = model(X_train)                # forward pass
    loss   = criterion(logits, y_train)    # compute loss

    optimizer.zero_grad()                  # zero grads (same rule as before!)
    loss.backward()                        # backprop
    optimizer.step()                       # update weights

    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            test_logits = model(X_test)
            preds       = test_logits.argmax(dim=1)   # pick highest logit
            acc         = (preds == y_test).float().mean() * 100
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Test Acc: {acc:.1f}%")

# ── 5. Final evaluation ──────────────────────────────────────────
model.eval()
with torch.no_grad():
    preds = model(X_test).argmax(dim=1)
    acc   = (preds == y_test).float().mean() * 100
    print(f"\nFinal accuracy: {acc:.1f}%")

    # Per-class breakdown
    for i, name in enumerate(iris.target_names):
        mask    = y_test == i
        correct = (preds[mask] == y_test[mask]).sum()
        total   = mask.sum()
        print(f"  {name:12s}: {correct}/{total}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── 1. Load & prep data ──────────────────────────────────────────
iris = load_iris()
X, y = iris.data, iris.target          # X: (150, 4)  y: 0/1/2

scaler = StandardScaler()
X = scaler.fit_transform(X)            # zero mean, unit variance

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# numpy → torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)    # long for CrossEntropyLoss
y_test  = torch.tensor(y_test,  dtype=torch.long)

# ── 2. Model ─────────────────────────────────────────────────────
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16),     # 4 features → 16 hidden
            nn.ReLU(),
            nn.Linear(16, 16),    # 16 → 16
            nn.ReLU(),
            nn.Linear(16, 3)      # 16 → 3 classes (raw logits)
        )

    def forward(self, x):
        return self.net(x)        # no softmax here — CrossEntropyLoss handles it

model = IrisNet()

# ── 3. Loss & optimizer ──────────────────────────────────────────
criterion = nn.CrossEntropyLoss()          # expects raw logits + class indices
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ── 4. Training loop ─────────────────────────────────────────────
for epoch in range(200):
    model.train()

    logits = model(X_train)                # forward pass
    loss   = criterion(logits, y_train)    # compute loss

    optimizer.zero_grad()                  # zero grads (same rule as before!)
    loss.backward()                        # backprop
    optimizer.step()                       # update weights

    if (epoch + 1) % 20 == 0:
        model.eval()
        with torch.no_grad():
            test_logits = model(X_test)
            preds       = test_logits.argmax(dim=1)   # pick highest logit
            acc         = (preds == y_test).float().mean() * 100
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Test Acc: {acc:.1f}%")

# ── 5. Final evaluation ──────────────────────────────────────────
model.eval()
with torch.no_grad():
    preds = model(X_test).argmax(dim=1)
    acc   = (preds == y_test).float().mean() * 100
    print(f"\nFinal accuracy: {acc:.1f}%")

    # Per-class breakdown
    for i, name in enumerate(iris.target_names):
        mask    = y_test == i
        correct = (preds[mask] == y_test[mask]).sum()
        total   = mask.sum()
        print(f"  {name:12s}: {correct}/{total}")